# 01. Google AI Studio で始める Hello World

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/google-genai/ja/01_ai_studio_quickstart.ipynb)

本クックブックでは `drawlib.toml` の `[jupyter].setup_packages` 設定により、生成されるすべてのノートブック先頭に `!pip install -q google-genai` のみが自動挿入されるため、Google Colab 上で高速に起動でき手動パッケージ追加も不要です。Git リポジトリへ認証情報をハードコードせずに安全に Gemini を呼び出すため、**Google Cloud プロジェクト**に紐付いた API キー（`GEMINI_API_KEY`）を **Google AI Studio** で発行し、Colab 左サイドバーの **シークレット（`🔑` アイコン）** に保存して **「ノートブックからのアクセス（Notebook access）」** を ON に設定します。

![図1-1: Google AI Studio APIキー発行と Colab シークレットによる最短実行フロー](https://examples.drawlib.com/google-genai/images/01_ai_studio_quickstart/ch01_aistudio_workflow_ja.png)

*図1-1: Google AI Studio APIキー発行と Colab シークレットによる最短実行フロー*

## 1. Colab シークレットを使った最小構成の Hello World (`userdata` + `genai.Client`)

Google Colab で最初のコードセルを実行する前に、以下の手順（初回のみ）でシークレットを設定します。

1. **Google AI Studio**（`aistudio.google.com/apikey`）を開き、**「API キーを作成（Create API key）」** から既存または新規の **Google Cloud プロジェクト**を選択してキーを発行します。
2. Google Colab 左サイドバーの **シークレット（`🔑`）** アイコンを開き、名前を `GEMINI_API_KEY` としてキーを貼り付け、**「ノートブックからのアクセス」** トグルを ON にします。
3. セル先頭で `MODEL = "gemini-3.8-flash"` を定義して利用モデルを分かりやすく指定し、`userdata.get("GEMINI_API_KEY")` で読み込んだキーを `genai.Client` に渡します。

In [ ]:
!pip install -q google-genai

In [ ]:
MODEL = "gemini-3.8-flash"

from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model=MODEL,
    contents="こんにちは！",
)
print(response.text)

`genai.Client` は `google-genai` SDK 全体で共通のエントリーポイントです。本章で学ぶ `client.models.*` や `client.chats.*` の呼び出しコードは、AI Studio の API キー認証でも Google Cloud Vertex AI の IAM 認証でも一切書き換えることなくそのまま動作します。

## 2. Google Cloud プロジェクト連携：無料枠（Free Tier）と従量課金枠（Paid Tier）

Google AI Studio で発行されるすべての API キーは、背後にある **Google Cloud プロジェクト**に紐付いています。普段から Google Cloud を利用している開発者は、AI Studio のキー作成ダイアログで既存のプロジェクトを選択できるため、API の有効化（`generativelanguage.googleapis.com`）やクォータ監視、請求管理を一元化できます。

- **無料枠 — Free Tier（課金未設定プロジェクト）**: 選択した Google Cloud プロジェクトに **Cloud Billing（請求先アカウント）が紐付いていない場合**、API キーは完全無料の Free Tier として動作します。ハッカソンや学習・プロトタイピングに最適ですが、レート制限（RPM/TPM/RPD）は低めに設定されており、入出力データが Google 製品の品質改善に利用される場合があります（機密データの送信は避けてください）。
- **従量課金枠 — Paid Tier（Cloud Billing 有効化済み）**: 同じ Google Cloud プロジェクトで **Cloud Billing** を有効化すると、同じ `GEMINI_API_KEY` が自動的に従量課金枠（Pay-as-You-Go）へアップグレードされます。Python コードを1行も変更することなく、本番ワークロード向けの高い RPM/TPM クォータと**エンタープライズ水準のデータプライバシー（プロンプトや出力が Google のモデル学習に使用されない保証）**が適用されます。

![図1-2: Google AI Studio 無料枠と Cloud Billing 従量課金枠（Paid Tier）の比較](https://examples.drawlib.com/google-genai/images/01_ai_studio_quickstart/ch01_billing_tiers_ja.png)

*図1-2: Google AI Studio 無料枠と Cloud Billing 従量課金枠（Paid Tier）の比較*

## 3. `GenerateContentConfig` とトークン消費量（`usage_metadata`）の確認

モデルの振る舞い（`system_instruction`）や出力の再現性（`temperature`）を制御するには、`types.GenerateContentConfig` を `generate_content` に渡します。また、レスポンスの `response.usage_metadata` からは、入力プロンプト（`prompt_token_count`）、内部思考（`thoughts_token_count`）、最終回答（`candidates_token_count`）の正確なトークン数を取得できます。

In [ ]:
from google.genai import types

config = types.GenerateContentConfig(
    system_instruction="1つの短い文で答えてください。",
    temperature=0.2,
)

response = client.models.generate_content(
    model=MODEL,
    contents="Pythonとは何ですか？",
    config=config,
)
print(response.text)

usage = response.usage_metadata
print(f"Prompt tokens:     {usage.prompt_token_count or 0}")
print(f"Thoughts tokens:   {usage.thoughts_token_count or 0}")
print(f"Candidates tokens: {usage.candidates_token_count or 0}")

`gemini-3.8-flash` では内部推論に費やされたトークン数が `thoughts_token_count` として独立して記録されるため、入出力サイズと思考コストの内訳を `usage_metadata` から一目で把握できます。

### まとめと次のステップ

- **高速かつセキュアな実行環境**: `google-genai` のみの軽量セットアップと Colab シークレットによる `GEMINI_API_KEY` 管理を組み合わせることで、Git への鍵漏洩を防ぎつつ、Google Cloud プロジェクトの無料枠・従量課金枠を素早く使い分けられます。
- **次章（Chapter 02）のプレビュー**: 続く第2章では、後続の `gemini-3.8-flash` 呼び出しコードを1行も変えずに、`genai.Client` の初期化だけを **Google Cloud Vertex AI**（`vertexai=True`・IAM 認証）へ切り替える方法を解説します。